# 700 — Program Signature Construction

## Objective

Construct and validate the frozen Phase 7 program-query representations and the outcome-blind preparatory objects required by the prespecified conditional-null and benchmark designs before notebook 701 connectivity inference begins.

Notebook 700 is a preparatory and computational-calibration notebook. It does not generate perturbagen rankings, observed program–perturbagen connectivity results, conditional-randomization p-values, multiplicity results, compound prioritization, or mechanism-of-action summaries.

## Analytical boundary

This notebook consumes the registered frozen Phase 4, Phase 3, and notebook-110 handoffs as authoritative inputs.

It may use the frozen LINCS Level 5 matrix only for prospectively authorized outcome-blind quantities, including gene-level perturbational dispersion and the namespace-3 computational benchmark. It must not apply the real frozen program queries to the observed perturbational matrix in a way that computes or exposes real program–perturbagen connectivity.

The notebook does not:

* alter frozen Phase 4 program membership, weights, sign, or identity;
* use Phase 5 or Phase 6 evidence for query construction, eligibility, prioritization, or rescue;
* inspect compound connectivity or rank perturbagens;
* execute the primary inference reserved for notebook 701; or
* select the final Monte Carlo `Bmax` using observed scientific results.

The resistance-like orientation used by Phase 7 is derived only from frozen upstream Phase 3/4 evidence under the governing Phase 7 contract. It is stored separately from the unchanged Phase 4 gene weights.

## Execution status

The initial implementation checkpoint below is restricted to registered-input resolution, frozen query construction, resistance-like orientation validation, and fixed CMap-style query membership.

No stable `phase7.700.*` artifact is persisted in this checkpoint. Outcome-blind MAD stratification, the pre-benchmark resource envelope, benchmark execution, final `Bmax` freeze, artifact publication, and registry closure follow only after the preceding cells have been executed and inspected.


In [ ]:
# =============================================================================
# Imports
# =============================================================================

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.paths import (
    project_relative_path,
)


In [ ]:
# =============================================================================
# Resolve frozen Phase 7 preparation inputs
# =============================================================================

artifact_registry = load_artifact_registry()

INPUT_ARTIFACT_IDS = (
    "phase4.401.consensus_transcriptomic_program_catalog",
    "phase4.401.consensus_transcriptomic_gene_weights",
    "phase3.311.program_robustness_summary",
    "phase1.110.raw_input_inventory",
    "phase1.110.program_representation_fidelity",
    "phase1.110.bing_gene_handoff",
    "phase1.110.landmark_gene_handoff",
    "phase1.110.primary_signature_handoff",
    "phase1.110.primary_perturbagen_coverage",
)

input_paths = {
    artifact_id: resolve_artifact_path(
        artifact_registry,
        artifact_id,
    )
    for artifact_id in INPUT_ARTIFACT_IDS
}

for artifact_id, path in input_paths.items():
    print(
        f"{artifact_id}: "
        f"{project_relative_path(path)}"
    )


In [ ]:
# =============================================================================
# Load frozen inputs required for query construction
# =============================================================================

program_catalog = pd.read_csv(
    input_paths[
        "phase4.401.consensus_transcriptomic_program_catalog"
    ]
)

phase4_gene_weights = pd.read_csv(
    input_paths[
        "phase4.401.consensus_transcriptomic_gene_weights"
    ]
)

phase3_robustness = pd.read_csv(
    input_paths[
        "phase3.311.program_robustness_summary"
    ]
)

representation_fidelity = pd.read_csv(
    input_paths[
        "phase1.110.program_representation_fidelity"
    ]
)

bing_gene_handoff = pd.read_csv(
    input_paths[
        "phase1.110.bing_gene_handoff"
    ]
)

landmark_gene_handoff = pd.read_csv(
    input_paths[
        "phase1.110.landmark_gene_handoff"
    ]
)

print("Program catalog shape:", program_catalog.shape)
print("Phase 4 gene weights shape:", phase4_gene_weights.shape)
print("Phase 3 robustness shape:", phase3_robustness.shape)
print("Representation fidelity shape:", representation_fidelity.shape)
print("BING handoff shape:", bing_gene_handoff.shape)
print("Landmark handoff shape:", landmark_gene_handoff.shape)


In [ ]:
# =============================================================================
# Validate frozen query membership and inherited Phase 4 weights
# =============================================================================

PROGRAM_IDS = (
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
)

expected_program_set = set(PROGRAM_IDS)

for label, frame in {
    "program_catalog": program_catalog,
    "bing_gene_handoff": bing_gene_handoff,
    "landmark_gene_handoff": landmark_gene_handoff,
}.items():
    observed = set(
        frame["consensus_program_id"]
        .dropna()
        .astype(str)
        .unique()
    )
    if observed != expected_program_set:
        raise RuntimeError(
            f"Unexpected program universe in {label}: {sorted(observed)}"
        )

bing_counts = (
    bing_gene_handoff
    .groupby("consensus_program_id")["gene_id"]
    .nunique()
    .reindex(PROGRAM_IDS)
)

landmark_counts = (
    landmark_gene_handoff
    .groupby("consensus_program_id")["gene_id"]
    .nunique()
    .reindex(PROGRAM_IDS)
)

if not bing_counts.eq(1495).all():
    raise RuntimeError(
        f"Unexpected BING gene counts: {bing_counts.to_dict()}"
    )

if not landmark_counts.eq(90).all():
    raise RuntimeError(
        f"Unexpected landmark gene counts: {landmark_counts.to_dict()}"
    )

phase4_weight_lookup = (
    phase4_gene_weights[
        [
            "consensus_program_id",
            "gene_symbol",
            "consensus_weight",
        ]
    ]
    .rename(
        columns={
            "consensus_weight": "phase4_consensus_weight",
        }
    )
)

weight_check = (
    bing_gene_handoff[
        [
            "consensus_program_id",
            "gene_symbol",
            "consensus_weight",
        ]
    ]
    .merge(
        phase4_weight_lookup,
        on=[
            "consensus_program_id",
            "gene_symbol",
        ],
        how="left",
        validate="one_to_one",
    )
)

if weight_check["phase4_consensus_weight"].isna().any():
    raise RuntimeError(
        "At least one frozen BING query gene is missing from the "
        "authoritative Phase 4 gene-weight artifact."
    )

if not np.array_equal(
    weight_check["consensus_weight"].to_numpy(dtype=np.float64),
    weight_check["phase4_consensus_weight"].to_numpy(dtype=np.float64),
):
    raise RuntimeError(
        "Notebook-110 BING weights do not exactly match the frozen "
        "Phase 4 gene weights."
    )

for program_id in PROGRAM_IDS:
    bing_landmark_ids = set(
        bing_gene_handoff.loc[
            bing_gene_handoff["consensus_program_id"].eq(program_id)
            & bing_gene_handoff["feature_space"].eq("landmark"),
            "gene_id",
        ]
    )
    landmark_ids = set(
        landmark_gene_handoff.loc[
            landmark_gene_handoff["consensus_program_id"].eq(program_id),
            "gene_id",
        ]
    )

    if bing_landmark_ids != landmark_ids:
        raise RuntimeError(
            f"Landmark membership mismatch for {program_id}."
        )

print("Frozen program universe: PASS")
print("BING membership: PASS", bing_counts.to_dict())
print("Landmark membership: PASS", landmark_counts.to_dict())
print("Inherited Phase 4 weights: exact match")


In [ ]:
# =============================================================================
# Derive and validate the frozen resistance-like orientation
# =============================================================================

program_source_metadata = (
    program_catalog[
        [
            "consensus_program_id",
            "cell_line_program",
            "orientation_multiplier",
        ]
    ]
    .rename(
        columns={
            "consensus_program_id": "program_id",
            "cell_line_program": "source_cell_line_program",
            "orientation_multiplier": "phase4_orientation_multiplier",
        }
    )
    .merge(
        phase3_robustness[
            [
                "program_id",
                "lineage_adjusted_rho",
                "robustness_category",
            ]
        ].rename(
            columns={
                "program_id": "source_cell_line_program",
                "lineage_adjusted_rho": "phase3_lineage_adjusted_rho",
                "robustness_category": "phase3_robustness_category",
            }
        ),
        on="source_cell_line_program",
        how="left",
        validate="one_to_one",
    )
    .sort_values("program_id")
    .reset_index(drop=True)
)

if program_source_metadata[
    [
        "phase4_orientation_multiplier",
        "phase3_lineage_adjusted_rho",
    ]
].isna().any().any():
    raise RuntimeError(
        "Incomplete frozen Phase 3/4 orientation metadata."
    )

program_source_metadata[
    "resistance_like_orientation_multiplier"
] = np.sign(
    program_source_metadata["phase4_orientation_multiplier"]
    * program_source_metadata["phase3_lineage_adjusted_rho"]
).astype("int8")

# This is an exact cross-check against the prospectively frozen Phase 7
# orientation table, not a second method for choosing orientation.
EXPECTED_RESISTANCE_LIKE_MULTIPLIERS = {
    "CONSENSUS_TX_01": -1,
    "CONSENSUS_TX_02": 1,
    "CONSENSUS_TX_03": 1,
}

observed_multipliers = (
    program_source_metadata
    .set_index("program_id")[
        "resistance_like_orientation_multiplier"
    ]
    .astype(int)
    .to_dict()
)

if observed_multipliers != EXPECTED_RESISTANCE_LIKE_MULTIPLIERS:
    raise RuntimeError(
        "Derived resistance-like orientation does not match the "
        "frozen Phase 7 contract."
    )

display(program_source_metadata)


In [ ]:
# =============================================================================
# Materialize deterministic continuous BING query definitions
# =============================================================================

program_query_definitions = (
    bing_gene_handoff[
        [
            "consensus_program_id",
            "gene_id",
            "gene_symbol",
            "feature_space",
            "gctx_gene_index",
            "consensus_weight",
        ]
    ]
    .rename(
        columns={
            "consensus_program_id": "program_id",
            "consensus_weight": "frozen_weight",
        }
    )
    .merge(
        program_source_metadata,
        on="program_id",
        how="left",
        validate="many_to_one",
    )
    .assign(
        gene_id=lambda data: pd.to_numeric(
            data["gene_id"],
            errors="raise",
        ).astype("int64"),
        gctx_gene_index=lambda data: pd.to_numeric(
            data["gctx_gene_index"],
            errors="raise",
        ).astype("int64"),
        primary_bing_member=True,
        landmark_sensitivity_member=lambda data: (
            data["feature_space"].eq("landmark")
        ),
    )
    .sort_values(
        [
            "program_id",
            "gene_id",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

program_query_definitions["query_gene_order"] = (
    program_query_definitions
    .groupby("program_id", sort=False)
    .cumcount()
    .astype("int64")
)

if not np.isfinite(
    program_query_definitions["frozen_weight"].to_numpy(
        dtype=np.float64
    )
).all():
    raise RuntimeError(
        "Program query contains a non-finite frozen weight."
    )

query_norms = (
    program_query_definitions
    .groupby("program_id", sort=False)["frozen_weight"]
    .apply(
        lambda values: np.linalg.norm(
            values.to_numpy(dtype=np.float64)
        )
    )
    .reindex(PROGRAM_IDS)
)

if not query_norms.gt(0).all():
    raise RuntimeError(
        f"Program query has zero norm: {query_norms.to_dict()}"
    )

query_summary = (
    program_query_definitions
    .groupby("program_id", as_index=False)
    .agg(
        bing_genes=("gene_id", "nunique"),
        landmark_genes=("landmark_sensitivity_member", "sum"),
        positive_weights=(
            "frozen_weight",
            lambda values: int((values > 0).sum()),
        ),
        negative_weights=(
            "frozen_weight",
            lambda values: int((values < 0).sum()),
        ),
    )
)

query_summary["l2_norm"] = (
    query_summary["program_id"].map(query_norms)
)

display(query_summary)


In [ ]:
# =============================================================================
# Freeze fixed CMap-style 150 / 150 query membership
# =============================================================================

program_query_definitions["cmap_style_arm"] = pd.Series(
    pd.NA,
    index=program_query_definitions.index,
    dtype="string",
)

for program_id in PROGRAM_IDS:
    program_mask = (
        program_query_definitions["program_id"].eq(program_id)
    )

    positive_candidates = (
        program_query_definitions.loc[
            program_mask
            & program_query_definitions["frozen_weight"].gt(0)
        ]
        .sort_values(
            [
                "frozen_weight",
                "gene_id",
            ],
            ascending=[False, True],
            kind="mergesort",
        )
    )

    negative_candidates = (
        program_query_definitions.loc[
            program_mask
            & program_query_definitions["frozen_weight"].lt(0)
        ]
        .sort_values(
            [
                "frozen_weight",
                "gene_id",
            ],
            ascending=[True, True],
            kind="mergesort",
        )
    )

    if len(positive_candidates) < 150:
        raise RuntimeError(
            f"{program_id} has fewer than 150 positive BING weights."
        )

    if len(negative_candidates) < 150:
        raise RuntimeError(
            f"{program_id} has fewer than 150 negative BING weights."
        )

    up_index = positive_candidates.head(150).index
    down_index = negative_candidates.head(150).index

    program_query_definitions.loc[
        up_index,
        "cmap_style_arm",
    ] = "up"

    program_query_definitions.loc[
        down_index,
        "cmap_style_arm",
    ] = "down"

program_query_definitions["cmap_style_member"] = (
    program_query_definitions["cmap_style_arm"].notna()
)

cmap_membership_summary = (
    program_query_definitions.loc[
        program_query_definitions["cmap_style_member"]
    ]
    .groupby(
        [
            "program_id",
            "cmap_style_arm",
        ],
        as_index=False,
    )
    .agg(
        genes=("gene_id", "nunique"),
    )
    .sort_values(
        [
            "program_id",
            "cmap_style_arm",
        ]
    )
    .reset_index(drop=True)
)

expected_cmap_counts = (
    cmap_membership_summary
    .pivot(
        index="program_id",
        columns="cmap_style_arm",
        values="genes",
    )
    .reindex(PROGRAM_IDS)
)

if not (
    expected_cmap_counts["up"].eq(150).all()
    and expected_cmap_counts["down"].eq(150).all()
):
    raise RuntimeError(
        "Fixed CMap-style query membership is not 150 / 150 "
        "for every frozen program."
    )

display(cmap_membership_summary)


In [ ]:
# =============================================================================
# Validate the initial outcome-blind query-construction checkpoint
# =============================================================================

checkpoint_summary = (
    program_query_definitions
    .groupby("program_id", as_index=False)
    .agg(
        bing_genes=("gene_id", "nunique"),
        landmark_genes=("landmark_sensitivity_member", "sum"),
        cmap_up_genes=(
            "cmap_style_arm",
            lambda values: int(values.eq("up").sum()),
        ),
        cmap_down_genes=(
            "cmap_style_arm",
            lambda values: int(values.eq("down").sum()),
        ),
        source_cell_line_program=(
            "source_cell_line_program",
            "first",
        ),
        phase4_orientation_multiplier=(
            "phase4_orientation_multiplier",
            "first",
        ),
        phase3_lineage_adjusted_rho=(
            "phase3_lineage_adjusted_rho",
            "first",
        ),
        phase3_robustness_category=(
            "phase3_robustness_category",
            "first",
        ),
        resistance_like_orientation_multiplier=(
            "resistance_like_orientation_multiplier",
            "first",
        ),
    )
)

if len(program_query_definitions) != 4485:
    raise RuntimeError(
        "Unexpected total number of BING query-definition rows."
    )

if program_query_definitions[
    [
        "program_id",
        "gene_id",
    ]
].duplicated().any():
    raise RuntimeError(
        "Duplicate program × gene rows in query definitions."
    )

print("Initial Notebook 700 query-construction checkpoint: PASS")
print("Program-query rows:", len(program_query_definitions))
print(
    "Observed LINCS connectivity computed in this checkpoint: NO"
)
print(
    "Stable phase7.700 artifacts persisted in this checkpoint: NO"
)

display(checkpoint_summary)


## Next implementation checkpoint

After the cells above have been executed and inspected, the next block will construct the outcome-blind gene-level perturbational-dispersion statistic and deterministic `feature_space × MAD_decile` stratification required by the frozen primary null.

That block will use the complete frozen 362,036-signature primary Level 5 universe and the full 10,174-gene BING space. It will not compute real program–perturbagen connectivity.

The pre-benchmark resource envelope and namespace-3 benchmark remain later steps. No benchmark timing may be measured before that resource envelope has been persisted prospectively.
